# 02 — Inference với DeepSeek-R1-Distill

## Mục tiêu
Chạy một checkpoint theo giao thức zero-shot trên dataset tiếng Việt đã kiểm tra bằng notebook 01. Mỗi câu nhận một user message riêng. Notebook điều phối module Python và kiểm tra đầu ra của lần chạy.

## Chuẩn bị trên Colab
Chọn runtime GPU NVIDIA. Đặt thư mục làm việc tại gốc bản sao dự án hoặc `notebooks/`. Chạy `python -m pip install -r requirements.txt` từ gốc dự án và khởi động lại runtime nếu Colab yêu cầu. Transformers được ghim ở 4.57.1 vì phiên bản này hỗ trợ Qwen2. Cấu hình ba checkpoint đều khai báo `Qwen2ForCausalLM`. Kiểm tra GPU thực tế trong pilot, nhất là khi dùng 4-bit hoặc 14B.

Sửa YAML để điền `dataset.path`, `inference.max_new_tokens` và một seed cho mỗi lần sinh. Nên ghi `dataset.version` và ghim revision của model/tokenizer trước test. Giá trị `dtype: null` dùng `float16`. Bắt đầu với `batch_size: 1` rồi đo VRAM trong pilot. Bật `enabled` của 14B nếu muốn chọn checkpoint đó. Fixture giả không được dùng cho inference.

Để tiếp tục sau khi Colab hủy runtime, đặt `OUTPUT_DIR` vào thư mục lưu trữ bền vững đã gắn, chẳng hạn thư mục trên Google Drive. Giữ bản YAML và dataset đã dùng. Chỉ chạy một phiên ghi cho mỗi `RUN_ID`.

In [ ]:
from pathlib import Path
import subprocess
import sys

cwd = Path.cwd()
root = next((path for path in (cwd, *cwd.parents) if (path / 'src/inference.py').is_file()), None)
if root is None:
    raise FileNotFoundError('Mở notebook từ bản sao dự án trên Colab.')

CONFIG_PATH = 'configs/experiment.yaml'
MODEL_ID = 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B'
RUN_ID = 'pilot-1p5b-01'
OUTPUT_DIR = None  # None dùng output.directory trong YAML.

## Phương pháp và chạy inference
Module áp dụng chat template của tokenizer cho prompt tiếng Việt, sinh theo batch cố định với padding bên trái và lưu riêng phần token sinh thêm. Mỗi batch hoàn tất được ghi thành một tệp JSONL. Khi chạy lại cùng `RUN_ID` và cùng cấu hình, module kiểm tra manifest và bỏ qua các batch đã lưu. Batch đang chạy lúc phiên ngắt có thể cần chạy lại.

Nếu GPU hết bộ nhớ, có thể tiếp tục với cấu hình cũ trên GPU đủ VRAM. Dùng `RUN_ID` mới khi muốn đổi batch size, dtype, lượng tử hóa hoặc chủ động chạy lại. Chạy từng model với `RUN_ID` riêng. Subprocess kết thúc sẽ trả bộ nhớ GPU cho runtime.

In [ ]:
command = [sys.executable, '-m', 'src.inference', '--config', CONFIG_PATH,
           '--model-id', MODEL_ID, '--run-id', RUN_ID]
if OUTPUT_DIR is not None:
    command.extend(['--output-dir', str(OUTPUT_DIR)])
subprocess.run(command, cwd=root, check=True)

## Kết quả và kiểm tra
Khi hoàn tất, `predictions.jsonl` chứa một dòng cho mỗi cặp câu hỏi và lần sinh. `manifest.json` lưu fingerprint dataset, revision model/tokenizer và cấu hình chạy. Token count tính cả EOS nếu có và bỏ padding sau EOS. `inference_time` tính bằng giây, là thời gian sinh của batch chia cho số câu trong batch. Đây là chi phí trung bình, không phải độ trễ đo riêng cho từng câu.

Kiểm tra số đầu ra và tỷ lệ chạm giới hạn token để diễn giải pilot. `prompt` lưu chuỗi chat thực tế, có thể chứa thẻ `<think>` do tokenizer thêm. `generated_response` chỉ chứa phần model sinh thêm. Notebook này chưa chấm đáp án.

In [ ]:
import json
import pandas as pd
import yaml
from IPython.display import display

config = yaml.safe_load((root / CONFIG_PATH).read_text(encoding='utf-8'))
run_dir = root / (OUTPUT_DIR or config['output']['directory']) / RUN_ID
manifest = json.loads((run_dir / 'manifest.json').read_text(encoding='utf-8'))
with (run_dir / 'predictions.jsonl').open(encoding='utf-8') as stream:
    predictions = pd.DataFrame(json.loads(line) for line in stream)
expected = manifest['question_count'] * len(manifest['inference']['seeds'])
print(f'Đầu ra: {len(predictions)}/{expected}')
print(f'Tỷ lệ chạm trần token: {predictions["stop_reason"].eq("max_new_tokens").mean():.1%}')
display(predictions[['question_id', 'generation_index', 'generated_token_count', 'stop_reason']].head())